<a href="https://colab.research.google.com/github/callsourav1979-personal/Assignments_HAAI-/blob/main/CV__Sorting_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
cv_folder = Path("/content/cvs")
cv_folder.mkdir(exist_ok = True)

print(cv_folder)

/content/cvs


In [2]:
from pathlib import Path
jd_folder = Path("/content/jds")
jd_folder.mkdir(exist_ok = True)

# **Import & Test Libraries**

In [3]:
!pip install -q pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.9 MB/s eta 0:00:00


In [4]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Setting up poppler-utils (22.02.0-2ubuntu0.13) ...
Processing triggers for man-db (2.10.2-1) ...


In [5]:
import torch
import sys
import pypdf
import docx
import pytesseract
from pdf2image import convert_from_path
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:" , device)

print("Python version:" , sys.version)
print("PyTorch version:" , torch.__version__)
print("CUDA available:" , torch.cuda.is_available())
print("PyPdf version:" , pypdf.__version__)
print("python-docx version:" , docx.__version__)
print("PyTesseract version:" , pytesseract.__version__)

if torch.cuda.is_available():
    print("CUDA version:" , torch.version.cuda)
    print("GPU device name:" , torch.cuda.get_device_name(0))
    print("GPU Memory:" , round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),"GB")
else:
  print("Running on CPU")

Using Device: cuda
Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch version: 2.11.0+cu128
CUDA available: True
PyPdf version: 6.16.2
python-docx version: 1.2.0
PyTesseract version: 0.3.13
CUDA version: 12.8
GPU device name: Tesla T4
GPU Memory: 14.56 GB


In [6]:
from pypdf import PdfReader
from docx import Document
from pathlib import Path

def extract_text_from_file(file_path):
    """
    Extract text from PDF or DOCX files.

      For PDF:
          Extracts texts from all pages.

      For DOCX:
          Extracts texts from normal paragraphs and tables.

      Parameters :
           file_path(str): Path to the PDF or DOCX file.
      Returns :
           str: Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    #---------------------------------
    # PDF
    #---------------------------------
    if extension == '.pdf':
        reader = PdfReader(str(path))

        pages = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
              pages.append(text)
        return '\n'.join(pages).strip()
    #---------------------------------------
    # DOCX
    #---------------------------------------
    elif extension == '.docx':
        document = Document(str(path))

        sections = []
        # Extract normal paragraphs
        for paragraph in document.paragraphs:
            text =  paragraph.text.strip()
            if text:
              sections.append(text)

        # Extract tables
        for table in document.tables:
            sections.append("\n[TABLE ]")
            for row in table.rows:
                row_cells = []
                for cell in row.cells:
                    cell_text = cell.text.strip()
                    if cell_text:
                      row_cells.append(cell_text)
                #Combine cells in the same row
                if row_cells:
                   sections.append(" | ".join(row_cells))
            sections.append("[/TABLE]")
        #Combine paragraphs and tables
        extracted_text = "\n".join(sections).strip()

        return extracted_text

    else:
      raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")


In [7]:
from pdf2image import convert_from_path
import pytesseract
from pathlib import Path

def extract_text_from_pdf_ocr(file_path):
    """
    Extract text from scanned/image based PDF files using OCR

     Parameters :
           file_path(str): Path to the PDF file.
      Returns :
           str: OCR Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    if extension != '.pdf':
      raise ValueError(f"This function supports PDF files only.")

    # Convert PDF pages into images
    pages = convert_from_path(str(path), dpi=300)

    extracted_pages = []

    # Process each page
    for page_number , page_image in enumerate(pages, start =1):
        print(f"Processing page {page_number}/{len(pages)}")
        #Run OCR
        text = pytesseract.image_to_string(page_image,config="--psm 6")

        #Remove unnessary whitespace
        text = text.strip()

        #Store page seperately
        page_text = (f"\n---PAGE {page_number} ---\n" f"{(text)}")

    extracted_pages.append(page_text)

    # Combine all pages
    final_text = "\n".join(extracted_pages)

    return final_text.strip()

## **`Wrapper Program`**

In [8]:
from pathlib import Path

def extract_document_text(file_path):
  """
  Main document-extraction wrapper
  Automatically selects the appropriate extraction method based on the file type
  and available text.

    Parameters:
       file_path (str) : Path to the resume file

    Returns:
       str : Extracted text from the resume
  """
  path = Path(file_path)

  #1. Check whether the file exists
  if not path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

  #2. Check supported file types
  extension = path.suffix.lower()
  if extension not in ['.pdf' , '.docx']:
    raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")

  #3 Handle PDF
  if extension == '.pdf':
    print(f"\nProcessng PDF: {path.name}")

    #First try normal PDF text extraction
    text = extract_text_from_file(path)

    #Check whether meaningful text was extracted
    if text and len(text.strip()) >= 100 :
      print("Text layer detected. Using standard PDF extraction.")
      return text.strip()
    if len(text.strip()) < 100 :
      print("Little or no text detected . Swtching to OCR ...")
      text = extract_text_from_pdf_ocr(path)
      return text.strip()

  #4 Handle DOCX
  elif extension == '.docx':
    print(f"\nProcessing DOCX: {path.name}")
    text = extract_text_from_file(path)
    return text.strip()

## **Extract Multiple CV's**

In [9]:
from pathlib import Path

def extract_multiple_cvs(cv_folder):
    """
    Extract text from all supported CV files in a folder.

    Supported formats:
      - PDF
      - DOCX

    Parameters:
       cv_folder (str) : Path to the folder containing CVs

    Returns:
       dict: Dictionary containing filename and extracted text
    """

    folder = Path(cv_folder)
    if not folder.exists():
        raise FileNotFoundError(f"CV Folder not found: {cv_folder}")

    if not folder.is_dir():
        raise ValueError(f"Path is not a directory: {cv_folder}")

    # Find all PDF and DOCX files
    cv_files = sorted(
                       [ file
                         for file in folder.iterdir()
                         if file.is_file() and file.suffix.lower() in [".pdf" , ".docx"]
                        ]
                      )
    if not cv_files:
      raise ValueError(f"No PDF or DOCX files found in :  {cv_folder}")

    cv_data = {}

    for cv_file in cv_files:
      print("="*60)
      print(f"Processing CV: {cv_file.name}")
      print("="*60)

      try:
        text = extract_document_text(cv_file)
        cv_data[cv_file.name] = text
        print(f"Characters Extracted: {len(text)}")
      except Exception as e:
        print(f"Error processing {cv_file.name}: {e}")
        cv_data[cv_file.name] = ""

    return cv_data

# **Install Transformer & Model**

In [10]:
!pip install -q transformers torch accelerate


# **Model-1 Loading**

In [11]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch

model_name1 = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name1,trust_remote_code=True)
model=AutoModelForCausalLM.from_pretrained(model_name1,
                                           torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                                           trust_remote_code=True,device_map="auto")

model1=model.to(device)
print("Model loaded Successfully")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded Successfully


# **Model-2 Loading**

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch

model_name2 = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer2=AutoTokenizer.from_pretrained(model_name2)
model2=AutoModelForCausalLM.from_pretrained(model_name2,
                                            torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                                            device_map="auto")

#model2=model2.to(device)
print("Model loaded Successfully")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Model loaded Successfully


In [ ]:
jd_text= all_jd["JD-Technical_Architect.pdf"]
print(jd_text[:10000])

# **Clean & Validate JSON Output**

In [21]:
import json

def clean_and_validate_json(response):
  response = response.strip()
  # Remove opening markdown fence
  if response.startswith("```json"):
    response = response[len("```json"):].strip()
  elif response.startswith("```"):
    response = response[len("```"):].strip()

  # Remove closing markdown fence
  if response.endswith("```"):
    response = response[:-3].strip()

  # Validate JSON
  return json.loads(response)

# **JD to JSON Conversion PROMPT**

In [36]:

def convert_jd_to_json(jd_text):

  jd_prompt = """
  DOCUMENT TYPE: JOB DESCRIPTION (JD)

  You are a precise recruitment information extraction assistant.

  Your task is to extract information ONLY from the provided JOB DESCRIPTION.
  Do not use outside knowledge. Do not infer, assume, invent, or fabricate information.

  IMPORTANT:
  This is a JOB DESCRIPTION, NOT a candidate CV.

  Return ONLY valid JSON.
  Do not return explanations, comments, markdown, keywords, or any text outside the JSON object.

  Use EXACTLY this JSON structure:

  {
    "job_title": "",
    "skills": [],
    "experience": [],
    "education": [],
    "responsibilities": []
  }

  FIELD DEFINITIONS:

  1. job_title
    Extract the exact job/position title stated in the JD.

  2. skills
    Extract technical skills, technologies, frameworks, platforms, tools,
    methodologies, architectural patterns, and certifications explicitly
    mentioned as required or preferred in the JD.

  Do not invent related technologies that are not explicitly mentioned.

  3. experience
    IMPORTANT: For a JOB DESCRIPTION, "experience" means
    EMPLOYER-STATED EXPERIENCE REQUIREMENTS.

  It does NOT mean candidate employment history.

  Extract experience requirements such as:
  - minimum total years of professional/software engineering experience
  - minimum years of architectural or technical leadership experience
  - required years of experience in a particular area
  - required experience with particular types of systems
  - required experience with methodologies or environments

  Preserve the meaning and wording of the JD as closely as possible.

  For example, if the JD says:
  "Minimum of 8+ years of total software engineering experience"

  then return:

  "experience": [
    "Minimum of 8+ years of total software engineering experience"
  ]

  If the JD says:
  "at least 3+ years acting in a dedicated Architectural or Tech Lead capacity"

  then return:

  "experience": [
    "At least 3+ years acting in a dedicated Architectural or Tech Lead capacity"
  ]

  NEVER convert an experience requirement into a fake employment-history object.

  DO NOT create fields such as:
  "title", "company", "location", "duration", or "description"
  inside the JD experience array.

  4. education
  Extract ONLY education requirements explicitly stated in the JD.

  For example, if the JD says:
  "Bachelor's or Master's degree in Computer Science, Software Engineering,
  or an equivalent technical field."

  return the relevant education requirement using the information actually
  present in the JD.

  DO NOT invent:
  - university names
  - graduation years
  - degree dates
  - fields of study not stated in the JD
  - candidate education details

  5. responsibilities
  Extract responsibilities, duties, activities, and expectations explicitly
  stated in the JD.

  Preserve the meaning of the JD.

  CRITICAL RULES:

  1. Extract information ONLY from the provided JD.
  2. Do NOT use outside knowledge.
  3. Do NOT infer or assume missing information.
  4. Do NOT invent companies, universities, candidates, dates, job histories,
     qualifications, or other information.
  5. Do NOT create candidate employment history from a JD.
  6. For a JD, the "experience" field contains EMPLOYER REQUIREMENTS,
     not candidate work history.
  7. For a JD, experience items must be strings, not employment-history objects.
  8. For a JD, do not create "Company A", "Company B", "University X",
     or similar placeholder/fabricated values.
  9. If information is not available, return [] for list fields and "" for
     the job_title field.
  10. Do not output values such as "Unknown", "Not specified",
     "Not mentioned", or "None".
  11. Do not include personal information that is not relevant to the
     requested extraction.
  12. Do not create a "keywords" field.
  13. Do not add any fields to the JSON structure.
  14. Return ONLY the JSON object..

  NOW EXTRACT THE INFORMATION FROM THIS JOB_DESCRIPTION:
  ----------------------------JOB DESCRIPTION START----------------------------------
  """ + jd_text + """

  ----------------------------JOB DESCRIPTION END----------------------------------

  """

  messages = [
      {"role": "system", "content": "You are a precise recruitment assistant that extracts information from CV"},
      {"role": "user", "content": jd_prompt}
    ]

  text= tokenizer.apply_chat_template(messages , tokenize=False , add_generation_prompt= True)

  inputs = tokenizer(text, return_tensors="pt").to(device)


  with torch.no_grad():
    outputs = model1.generate(**inputs,max_new_tokens=500,do_sample=False)

  jd_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)

  return clean_and_validate_json(jd_response)


## **READ JD from Folder**

In [39]:
from pathlib import Path

def read_jd_from_folder(jd_folder):
 jd_folder = Path(jd_folder)

 # Find the first PDF or DOCX file
 jd_files = list(jd_folder.glob("*.pdf")) + list(jd_folder.glob("*.docx"))

 if not jd_files:
    raise FileNotFoundError("No JD file found in the folder.")

 jd_file = jd_files[0]

 print(f"JD file found: {jd_file}")

  # Use your existing extraction function
 jd_text = extract_document_text(str(jd_file))

 jd_json = convert_jd_to_json(jd_text)
 #print(json.dumps(jd_json,indent=2))

 return jd_json

In [ ]:
read_jd_from_folder("/content/jds")

# **CV to JSON Conversion PROMPT**

In [38]:

def convert_cv_to_json(cv_text):

  cv_prompt = """
  You are an expert recruitment assistant.

  Your task is to analyze the following CV/Resume and convert the information explicitly stated in it into a structured JSON format.

  JOB DESCRIPTION :
  """ + cv_text + """

  Extract the following information:
  1. Candidate Name
  2. Job title or professional title, if explicitly stated
  3. Skills explicitly mentioned in the CV
  4. Work experience explicitly mentioned in the CV
  5. Education/qualifications explicitly mentioned in the CV
  6. Job responsibilities , duties , projects , or work activities explicitly mentioned in the CV


  Return ONLY vaid JSON in exactly this format:

  {
    "candidate_name": "",
    "job_title": "",
    "skills": [],
    "experience": [],
    "education": [],
    "responsibilities": []
  }

  IMPORTANT RULES:

  1. Extract information only from the CV. Do not use outside
   knowledge or assumptions.

  2. Preserve the original meaning and wording of the Job Description
   as closely as possible.

  3. Do NOT classify anything as required, preferred, or other at this stage.

  4. "skills" must contain actual skills, abilities, knowledge, tools,
   technologies, software, programming languages, or competencies
   explicitly mentioned in the Job Description.

  5. "experience" must contain explicitly stated experience requirements
   or experience statements.

  6. "education" must contain explicitly stated educational qualifications,
   degrees, diplomas, certifications, or fields of study.

  7. "responsibilities" must contain the actual duties and responsibilities
   stated in the Job Description.

  8. Do NOT convert responsibilities into skills.

  9. Do NOT convert education into skills.

  10. Do NOT convert experience into skills.

  11. Keep responsibilities as complete statements. Do not unnecessarily
    summarize or omit important responsibilities.

  12. If information is not present, return an empty list [].

  13. NEVER output "None specified", "Not specified", "Not mentioned",
    "Unknown", or similar text. Use [] instead.

  14. Return ONLY the JSON object in exactly the structured mentioned . Do not provide explanations,
    comments, markdown, or text outside the JSON.

  15. Do NOT add any fields that are not present in the JSON structure provided. Do NOT generate keywords , summary , profile or
    any additional fields.

  16. Stop generating immediately after the closing } of the JSON object.

  OUTPUT LIMITS:

  - Do not reproduce the CV verbatim.
  - Summarize each experience entry concisely.
  - Maximum 5 responsibilities per job.
  - Maximum 15 skills.
  - Maximum 5 education entries.
  - Do not repeat skills.
  - Do not repeat responsibilities.
  - Do not include explanations outside the JSON.
  - Return ONLY the JSON object.
  - Stop immediately after the closing }.
  - Do not copy entire paragraphs from the CV.
  - Extract only the information required by the JSON schema.
 """

  messages = [
      {"role": "system", "content": "You are a precise recruitment assistant that extracts information from CV"},
      {"role": "user", "content": cv_prompt}
    ]

  text= tokenizer.apply_chat_template(messages , tokenize=False , add_generation_prompt= True)

  inputs = tokenizer(text, return_tensors="pt").to(device)


  with torch.no_grad():
    outputs = model1.generate(**inputs,max_new_tokens=1200,do_sample=False)

  cv_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True
                                 )

  return  clean_and_validate_json(cv_response)

#print("LLM Response:" , cv_response.strip())


# **READ CV's from Folder & Convert to JSON**

In [ ]:
cv_folder ="/content/cvs"
all_cvs = extract_multiple_cvs(cv_folder)
print("Total CVs processed:" , len(all_cvs))

all_cv_json ={}

for filename , cv_text in all_cvs.items():
  print(f"Processing: , {filename}")

  try:
    cv_json = convert_cv_to_json(cv_text)
    all_cv_json[filename] = cv_json

    print(" Sucessfully converted")

  except Exception as e:
      print(f"Error processing {filename}: {e}")

  print("Total CVs converted: " , len(all_cv_json))

In [ ]:
import re
from difflib import SequenceMatcher


# ---------------------------------------------------------
# 1. TEXT NORMALIZATION
# ---------------------------------------------------------

def normalize(text):
    if not text:
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+#./ -]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ---------------------------------------------------------
# 2. FLATTEN CV / JD SKILLS
# ---------------------------------------------------------

def get_skills(data):
    skills = data.get("skills", [])

    if not isinstance(skills, list):
        return []

    result = []

    for skill in skills:
        if isinstance(skill, str):
            skill = skill.strip()

            if skill:
                result.append(skill)

    # remove duplicates while preserving order
    unique = []
    seen = set()

    for skill in result:
        key = normalize(skill)

        if key not in seen:
            seen.add(key)
            unique.append(skill)

    return unique


# ---------------------------------------------------------
# 3. DIRECT / FUZZY SKILL MATCHING
# ---------------------------------------------------------
def skill_matches(jd_skills, cv_skills):

    matched = []
    missing = []

    cv_normalized = {
        normalize(skill): skill
        for skill in cv_skills
        if isinstance(skill, str) and skill.strip()
    }

    for jd_skill in jd_skills:

        if not isinstance(jd_skill, str) or not jd_skill.strip():
            continue

        jd_norm = normalize(jd_skill)

        found = False

        # 1. EXACT MATCH ONLY
        if jd_norm in cv_normalized:
            matched.append(jd_skill)
            found = True

        # 2. FUZZY MATCH ONLY FOR LONGER SKILL NAMES
        if not found and len(jd_norm) >= 4:

            for cv_norm in cv_normalized:

                if len(cv_norm) < 4:
                    continue

                similarity = SequenceMatcher(
                    None,
                    jd_norm,
                    cv_norm
                ).ratio()

                if similarity >= 0.85:
                    matched.append(jd_skill)
                    found = True
                    break

        # 3. NOT FOUND
        if not found:
            missing.append(jd_skill)

    return matched, missing

# ---------------------------------------------------------
# 4. EXPERIENCE TEXT
# ---------------------------------------------------------

def get_experience_text(data):

    experiences = data.get("experience", [])

    if not isinstance(experiences, list):
        return ""

    text_parts = []

    for exp in experiences:

        if not isinstance(exp, dict):
            continue

        position = exp.get("position", "")
        company = exp.get("company", "")
        duration = exp.get("duration", "")

        text_parts.append(
            f"{position} {company} {duration}"
        )

        responsibilities = exp.get(
            "responsibilities",
            []
        )

        if isinstance(responsibilities, list):

            for responsibility in responsibilities:

                if isinstance(responsibility, str):
                    text_parts.append(responsibility)

    return " ".join(text_parts)


# ---------------------------------------------------------
# 5. EXPERIENCE MATCHING
# ---------------------------------------------------------

def calculate_experience_score(jd, cv):

    # JD required skills
    jd_skills = get_skills(jd)

    # Candidate's actual work experience
    cv_experience_text = get_experience_text(cv)

    if not jd_skills or not cv_experience_text:
        return 0

    jd_skills_normalized = [
        normalize(skill)
        for skill in jd_skills
    ]

    cv_text = normalize(cv_experience_text)

    # Check whether candidate's work experience
    # contains any of the JD's required skills
    matched_skills = []

    for skill in jd_skills_normalized:
        if skill in cv_text:
            matched_skills.append(skill)

    # No relevant technical/domain experience
    if not matched_skills:
        return 0

    # Relevant experience exists.
    # Score based on percentage of JD skills appearing
    # in the candidate's experience.
    score = (
        len(matched_skills) /
        len(jd_skills_normalized)
    ) * 100

    return min(100, round(score))

# ---------------------------------------------------------
# 6. EDUCATION MATCHING
# ---------------------------------------------------------

def calculate_education_score(jd, cv):

    jd_education = jd.get("education", [])
    cv_education = cv.get("education", [])

    # No education requirement in JD
    if not jd_education:
        return 100

    # Candidate has no education information
    if not cv_education:
        return 0

    # Convert JD education into text
    jd_text = normalize(" ".join(
        item if isinstance(item, str) else str(item)
        for item in jd_education
    ))

    # Convert CV education dictionaries into useful text
    cv_parts = []

    for item in cv_education:

        if isinstance(item, dict):
            cv_parts.append(str(item.get("degree", "")))
            cv_parts.append(str(item.get("field_of_study", "")))
        else:
            cv_parts.append(str(item))

    cv_text = normalize(" ".join(cv_parts))

    # --------------------------------
    # FIELD / SUBJECT MATCHING
    # --------------------------------

    technical_fields = [
        "computer science",
        "software engineering",
        "computer engineering",
        "information technology",
        "information systems",
        "electrical engineering",
        "electronics",
        "engineering"
    ]

    jd_fields = [
        field for field in technical_fields
        if field in jd_text
    ]

    cv_fields = [
        field for field in technical_fields
        if field in cv_text
    ]

    # If JD specifies a technical field,
    # candidate should have a related technical field
    if jd_fields:
        if not cv_fields:
            return 0

        if any(field in cv_fields for field in jd_fields):
            return 100

        return 50

    # --------------------------------
    # DEGREE MATCHING
    # --------------------------------

    if "master" in jd_text or "mba" in jd_text:
        if "master" in cv_text or "mba" in cv_text:
            return 100

    if "bachelor" in jd_text:
        if "bachelor" in cv_text or "master" in cv_text:
            return 100

    return 0
# ---------------------------------------------------------
# 7. OVERALL MATCHING
# ---------------------------------------------------------

def match_candidate(jd, cv):

    jd_skills = get_skills(jd)
    cv_skills = get_skills(cv)

    matched_skills, missing_skills = skill_matches(
        jd_skills,
        cv_skills
    )

    # Skill score
    if jd_skills:

        skill_score = round(
            len(matched_skills) /
            len(jd_skills) *
            100
        )

    else:
        skill_score = 0

    # Experience
    experience_score = calculate_experience_score(
        jd,
        cv
    )

    # Education
    education_score = calculate_education_score(
        jd,
        cv
    )

    # Overall
    overall_score = round(
        skill_score * 0.50 +
        experience_score * 0.30 +
        education_score * 0.20
    )

    # Recommendation
    if overall_score >= 80:
        recommendation = "Strong Match"

    elif overall_score >= 60:
        recommendation = "Good Match"

    elif overall_score >= 40:
        recommendation = "Moderate Match"

    elif overall_score >= 20:
        recommendation = "Weak Match"

    else:
        recommendation = "Poor Match"

    candidate_name = cv.get(
        "candidate_name",
        ""
    )

    return {
        "candidate_name": candidate_name,
        "overall_score": overall_score,
        "skill_match_score": skill_score,
        "experience_match_score": experience_score,
        "education_match_score": education_score,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "recommendation": recommendation
    }

In [ ]:
import json

jd_data = json.loads(jd_text)
cv_data = json.loads(cv_text)

print("JD type:", type(jd_data))
print("CV type:", type(cv_data))

result = match_candidate(jd_data, cv_data)

print(result)

JD type: <class 'dict'>
CV type: <class 'dict'>
{'candidate_name': 'David M. Vance', 'overall_score': 20, 'skill_match_score': 0, 'experience_match_score': 0, 'education_match_score': 100, 'matched_skills': [], 'missing_skills': ['Java (Spring Boot)', 'Go', 'Python', 'Node.js', '.NET Core Enterprise stacks', 'AWS', 'Azure', 'GCP', 'Terraform/OpenTofu (IaC)', 'Cloud Architecture Patterns', 'Docker', 'Kubernetes (EKS/GKE)', 'Istio Service Mesh', 'Helm Charts', 'PostgreSQL', 'MongoDB', 'Snowflake', 'Redis Clusters', 'Apache Kafka', 'RabbitMQ', 'RESTful APIs', 'gRPC', 'GraphQL Topologies', 'AWS Solutions Architect Professional', 'Google Professional Cloud Architect', 'Azure Solutions Architect Expert', 'TOGAF certification'], 'recommendation': 'Weak Match'}


In [ ]:
print("JD experience:", jd_data.get("experience"))
print("CV experience:", cv_data.get("experience"))

print("Experience score:",
      calculate_experience_score(jd_data, cv_data))

JD experience: ['Minimum of 8+ years of total software engineering experience', 'At least 3+ years acting in a dedicated Architectural or Tech Lead capacity']
CV experience: [{'company': 'Apex Global Solutions', 'position': 'Sales Director', 'duration': '2022 – Present', 'responsibilities': ['Direct all commercial activity and pipeline development for a $45M enterprise business segment.', 'Formulated an aggressive market-entry strategy that grew market share by 24% within 18 months.', 'Championed new incentive models for a team of 35 account executives, elevating average quota attainment from 72% to 91%.', 'Spearheaded contract negotiations for 12 Fortune 500 corporate accounts, securing $18M in recurring annual revenue commitments.']}, {'company': 'Vanguard Commercial Group', 'position': 'Regional Sales Manager', 'duration': '2016 – 2022', 'responsibilities': ['Overhauled the Eastern Region sales team, expanding territories and growing localized revenue from $12M to $28M.', 'Mentored 

# Matching **PROMPT**

In [ ]:
matching_prompt = f"""
You are an expert recruitment matching engine.

Your task is to compare ONE JOB DESCRIPTION (JD) with ONE CANDIDATE CV.

IMPORTANT:
You must judge the candidate ONLY from the information contained in the CV.
You must judge the requirements ONLY from the information contained in the JD.
DO NOT invent, assume, infer, or add skills, experience, certifications, education, or responsibilities that are not explicitly supported by the CV.

========================
JOB DESCRIPTION
========================

"""+jd_text+"""

========================
CANDIDATE CV
========================

"""+cv_text +"""

================================
SCORING AND MATCHING RULES:
================================

1. All scores must be integers from 0 to 100.
2. NEVER output a score below 0 or above 100.

3. skill_match_score:
   - Compare the candidate's actual skills against ONLY the skills explicitly
     required or strongly implied by the job description.
   - Do not invent skills that are not present in the job description.
   - Do not treat generic leadership, communication, management or business
     skills as technical skill matches unless the JD explicitly requires them.
   - If there are no meaningful technical skill matches, use 0.

4. experience_match_score:
   - Compare the candidate's actual work experience against the responsibilities
     and experience requirements in the JD.
   - Do not infer technical architecture experience from generic management or
     leadership experience.
   - If the candidate has no relevant experience, use 0.

5. education_match_score:
   - Compare only against education or certification requirements explicitly
     stated in the JD.
   - Do not give a high score merely because the candidate has a degree.
   - If the JD requires a technical certification that the candidate does not
     have, that requirement must be considered missing.

6. matched_skills:
   - Include ONLY skills that are actually present in the candidate CV AND
     relevant to the JD.
   - Maximum 8 items.
   - Every item must be unique.
   - Never repeat an item.

7. missing_skills:
   - Include ONLY skills, technologies, certifications or competencies that
     are actually required by the JD and are absent from the candidate CV.
   - NEVER invent missing skills.
   - Maximum 6 items.
   - Every item must be unique.
   - Never repeat an item.

8. relevant_experience:
   - Include ONLY experience from the CV that directly supports the JD.
   - Maximum 3 items.
   - If there is no directly relevant experience, return [].

9. overall_score:
   Calculate the overall score using:

   overall_score =
       (skill_match_score * 0.50) +
       (experience_match_score * 0.30) +
       (education_match_score * 0.20)

   Round the result to the nearest integer.

10. Recommendation:
   - 80-100: "Strong Match"
   - 60-79: "Good Match"
   - 40-59: "Moderate Match"
   - 20-39: "Weak Match"
   - 0-19: "Poor Match"

========================
OUTPUT FORMAT
========================

Return ONLY valid JSON.

Do NOT return:
- markdown
- ```json
- explanations before the JSON
- explanations after the JSON
- commentary
- analysis

Use EXACTLY these field names.
Do not change the spelling or capitalization.

{{
  "candidate_name": "",
  "overall_score": 0,
  "skill_match_score": 0,
  "experience_match_score": 0,
  "education_match_score": 0,
  "matched_skills": [],
  "missing_skills": [],
  "relevant_experience": [],
  "recommendation": ""
}}

========================
FINAL VALIDATION BEFORE ANSWERING
========================

Before producing the JSON, internally check:

A. Is every matched skill supported by the CV?
B. Is every missing skill actually required by the JD?
C. Did I accidentally copy JD skills into matched_skills?
D. Did I invent any candidate experience?
E. Did I give a high score only because of seniority or generic leadership?
F. Are duplicate missing skills removed?
G. Does the overall score agree with the skill and experience scores?
H. If this is a Sales CV against a Technical Architect JD, did I correctly identify it as a poor match if the technical evidence is absent?
I. Did I accidentally copy any sentence or responsibility from the JD into relevant_experience?
J. If the CV has no relevant experience, did I return an empty relevant_experience list?

IMPORTANT OUTPUT LIMITS:

- matched_skills: maximum 8 items
- missing_skills: maximum 6 items
- relevant_experience: maximum 3 items
- Each item must be unique.
- Never repeat an item.
- Keep each item short and concise.
- Do not provide explanations.
- If there is no relevant experience, return an empty list.
- If the candidate is a very poor match, keep relevant_experience empty.

Now produce ONLY the JSON result.
"""

messages2 = [
      {"role": "system", "content": "You are a precise recruitment matching assistant"},
      {"role": "user", "content": matching_prompt}
  ]

text2= tokenizer2.apply_chat_template(messages2 , tokenize=False , add_generation_prompt= True)

inputs2 = tokenizer2(text2, return_tensors="pt").to(device)


with torch.no_grad():
  outputs2 = model2.generate(**inputs2,max_new_tokens=500,do_sample=False,repetition_penalty=1.1,no_repeat_ngram_size=3 )

match_response = tokenizer2.decode(outputs2[0][inputs2["input_ids"].shape[1]:],skip_special_tokens=True)

print("LLM #2 Response:" , match_response.strip())




LLM #2 Response: ```json
{
    "candidateName": "Marcus Vance",
    'overallScore': 45,
    'skillMatchScore': null,
    "experienceMatchScore": null, 
    "educationMatchScore" : null,  
    "matchedSkills": ["EnterpriseAccountAcquisition", "RevenuepipelineOptimization", "KeyStakeholderRelations", "HighValueContractNegotiation"],
    "missingSkills" : ["RevenuePipelineOptimisation", "CrossFunctionalTeamLeadership", "ExecutivClientAdvisory", "PandPLeadership",  "SaleSDirectorResponsibilities"],
    'relevantExperience' : ["Directedallcommercialactivityandpipelinedevelopmentfortheaeventenerendesignsegment.", "Formulatetheanaggressivemarket-entrystrategytogrowmarketshareby24percentwithin18months.", "Speardealwithnewinsidetacrunchfor12Fortune50coaccountexecutives,elevatingaveragequotaattainmentsfrom72percentto91%."],
    "recommendedation" : "GoodMatch"
}```
